# Cross-scenario analysis — DemandForge × POMMES CLEVER 2050

Loads every solved scenario's diagnostics and computes comparative
tables/figures. Designed to run on inari where data + deps live, but also
works on a local Mac with the SMB mount.

Sections:
1. Config + imports
2. Load per-scenario diagnostics
3. Headline capacity per scenario
4. Comparative tables → tables/synthesis/*.csv
5. Comparative figures → figures/synthesis/*.{png,svg}
6. Nuclear investigation (H1/H2/H3)
6.5. Ramping diagnostic (ex-post ramp-rate check)
7. Corridor expansion analysis
8. System cost summary
9. Sanity assertions


## 1. Config + imports

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

_INARI = Path("/diskdata/cired/brigode/clever-work").exists()
WS = Path("/diskdata/cired/brigode/clever-work") if _INARI else Path("~/Desktop/clever-work").expanduser()
print(f"Workspace: {WS}  (inari={_INARI})")

def diag_path(scenario):
    if _INARI:
        return WS / "results" / "diagnostics" / scenario
    if scenario == "R0_v1":
        return WS / "R0_v1" / "diagnostics"
    return WS / f"results_{scenario}" / "diagnostics"

ALL_KNOWN = [
    "R0_v1", "policy_re", "policy_nuke", "R0_v1_nuke", "policy_re_noMin",
    "R0_v1_corr2x", "R0_v1_corr3x",
    "R0_v1_nuke_corr2x", "R0_v1_nuke_corr3x",
    "policy_re_corr2x", "policy_re_corr3x",
    "policy_nuke_corr2x", "policy_nuke_corr3x",
    "policy_re_noMin_corr2x", "policy_re_noMin_corr3x",
]
SCENARIOS = [s for s in ALL_KNOWN if (diag_path(s) / "solution_2050.nc").exists()]
print(f"Found solved scenarios ({len(SCENARIOS)}): {SCENARIOS}")

TABLES_OUT = WS / "tables" / "synthesis"
FIGS_OUT   = WS / "figures" / "synthesis"
TABLES_OUT.mkdir(parents=True, exist_ok=True)
FIGS_OUT.mkdir(parents=True, exist_ok=True)


## 2. Load per-scenario diagnostics

In [ ]:
scenarios_data = {}
for scen in SCENARIOS:
    d = diag_path(scen)
    try:
        scenarios_data[scen] = {
            "sol": xr.open_dataset(d / "solution_2050.nc"),
            "inp": xr.open_dataset(d / "input_dataset_2050.nc"),
        }
        print(f"  loaded {scen}")
    except Exception as e:
        print(f"  FAIL {scen}: {type(e).__name__}: {e}")
print(f"\n→ {len(scenarios_data)} scenarios in memory")


## 3. Headline capacity per scenario

In [ ]:
TECH_LABELS = {
    "Solar":                "solar_GW",
    "Wind_Onshore":         "wind_onshore_GW",
    "Wind_Offshore":        "wind_offshore_GW",
    "Nuclear":              "nuclear_GW",
    "Gas":                  "gas_GW",
    "Hydrogen_power_plant": "h2_pp_GW",
    "Waste":                "waste_GW",
    "electrolysis":         "electrolyser_GW",
    "Reservoir_Hydro_Plant":"reservoir_hydro_GW",
    "RoR_Hydro":            "ror_hydro_GW",
}

def headline(scen, d):
    out = {"scenario": scen}
    sol = d["sol"]
    cap = sol["operation_conversion_power_capacity"]
    techs_present = list(map(str, cap.coords["conversion_tech"].values))
    for tech, label in TECH_LABELS.items():
        if tech not in techs_present:
            out[label] = 0.0; continue
        out[label] = float(cap.sel(conversion_tech=tech).sum()) / 1000.0
    ls = sol["operation_load_shedding_power"]
    for r in ("hydrogen", "electricity"):
        if r in list(map(str, ls.coords["resource"].values)):
            out[f"shed_{r}_TWh"] = float(ls.sel(resource=r).sum()) / 1e6
    sec = sol["operation_storage_energy_capacity"]
    for stech in ("h2_storage", "Battery_4h", "Battery_1h", "Pumped_Hydro"):
        if stech in list(map(str, sec.coords["storage_tech"].values)):
            out[f"storeE_{stech}_TWh"] = float(sec.sel(storage_tech=stech).sum()) / 1e6
    if "annualised_totex" in sol.data_vars:
        out["totex_Geur"] = float(sol["annualised_totex"].sum()) / 1e9
    return out

rows = [headline(s, d) for s, d in scenarios_data.items()]
headline_df = pd.DataFrame(rows).set_index("scenario") if rows else pd.DataFrame()
if not headline_df.empty:
    if "R0_v1" in headline_df.index:
        base = headline_df.loc["R0_v1"]
        for col in [c for c in headline_df.columns if c.endswith("_GW") or c == "totex_Geur"]:
            headline_df[f"Δ_{col}_vs_R0_v1"] = headline_df[col] - base[col]
    headline_df.to_csv(TABLES_OUT / "headline_by_scenario.csv")
    print(headline_df[[c for c in headline_df.columns if not c.startswith("Δ_")]].round(2).to_string())


## 4. Comparative tables

In [ ]:
# 4.1 Capacity per country × tech
rows = []
for scen, d in scenarios_data.items():
    cap = d["sol"]["operation_conversion_power_capacity"]
    techs_present = list(map(str, cap.coords["conversion_tech"].values))
    for tech in TECH_LABELS:
        if tech not in techs_present: continue
        sel = cap.sel(conversion_tech=tech)
        for area in sel.coords["area"].values:
            v = float(sel.sel(area=area).sum()) / 1000.0
            if v > 0.001:
                rows.append({"scenario": scen, "country": str(area), "tech": tech, "GW": v})

cap_df = pd.DataFrame(rows)
if not cap_df.empty:
    cap_pivot = cap_df.pivot_table(index=["country", "tech"], columns="scenario", values="GW", fill_value=0)
    cap_pivot.to_csv(TABLES_OUT / "capacity_by_country_tech.csv")
    print(cap_pivot.head(25).to_string())


In [ ]:
# 4.2 Electrolyser redistribution per country
elec_rows = []
for scen, d in scenarios_data.items():
    cap = d["sol"]["operation_conversion_power_capacity"]
    if "electrolysis" not in list(map(str, cap.coords["conversion_tech"].values)): continue
    sel = cap.sel(conversion_tech="electrolysis")
    for area in sel.coords["area"].values:
        elec_rows.append({"scenario": scen, "country": str(area),
                          "electrolyser_GW": float(sel.sel(area=area).sum()) / 1000.0})
if elec_rows:
    elec_df = pd.DataFrame(elec_rows).pivot(index="country", columns="scenario", values="electrolyser_GW").fillna(0)
    base_col = "R0_v1" if "R0_v1" in elec_df.columns else elec_df.columns[0]
    for col in elec_df.columns:
        if col != base_col:
            elec_df[f"Δ_{col}_vs_{base_col}"] = elec_df[col] - elec_df[base_col]
    elec_df.to_csv(TABLES_OUT / "electrolyser_redistribution.csv")
    print(elec_df.sort_values(by=base_col, ascending=False).head(15).round(2).to_string())


## 5. Comparative figures

In [ ]:
if not headline_df.empty:
    vre_cols = [c for c in ("solar_GW", "wind_onshore_GW", "wind_offshore_GW") if c in headline_df.columns]
    disp_cols = [c for c in ("nuclear_GW", "gas_GW", "h2_pp_GW") if c in headline_df.columns]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    headline_df[vre_cols].plot.bar(stacked=True, ax=axes[0])
    axes[0].set_title("VRE capacity (GW)"); axes[0].grid(axis="y", alpha=0.3); axes[0].tick_params(axis="x", rotation=45)
    headline_df[disp_cols].plot.bar(stacked=True, ax=axes[1])
    axes[1].set_title("Dispatchable capacity (GW)"); axes[1].grid(axis="y", alpha=0.3); axes[1].tick_params(axis="x", rotation=45)
    if "electrolyser_GW" in headline_df.columns:
        headline_df["electrolyser_GW"].plot.bar(ax=axes[2], color="seagreen")
        axes[2].set_title("Electrolyser (GW)"); axes[2].grid(axis="y", alpha=0.3); axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout()
    out = FIGS_OUT / "headline_capacity_by_scenario"
    fig.savefig(f"{out}.png", dpi=120); fig.savefig(f"{out}.svg"); plt.close()
    print(f"→ wrote {out}.{{png,svg}}")


## 6. Nuclear investigation (H1 / H2 / H3)

In [ ]:
nuke_verdict = []
for scen, d in scenarios_data.items():
    inp = d["inp"]; sol = d["sol"]
    techs = list(map(str, inp.coords["conversion_tech"].values))
    if "Nuclear" not in techs:
        nuke_verdict.append({"scenario": scen, "in_techs": False,
                             "invest_min_GW": None, "invest_max_GW": None,
                             "deployed_GW": None, "verdict": "H1 (Nuclear not in model)"})
        continue
    inv_min = float(inp["conversion_power_capacity_investment_min"].sel(conversion_tech="Nuclear").sum()) / 1000
    inv_max = float(inp["conversion_power_capacity_investment_max"].sel(conversion_tech="Nuclear").sum()) / 1000
    deployed = float(sol["operation_conversion_power_capacity"].sel(conversion_tech="Nuclear").sum()) / 1000
    if inv_max <= inv_min + 0.01:
        verdict = "H1 (no headroom — min ≈ max)"
    elif deployed <= inv_min + 0.01:
        verdict = "H2 (uneconomic — at floor)"
    else:
        verdict = f"H3 (deployed {deployed:.1f} GW)"
    nuke_verdict.append({"scenario": scen, "in_techs": True,
                         "invest_min_GW": inv_min, "invest_max_GW": inv_max,
                         "deployed_GW": deployed, "verdict": verdict})

nuke_df = pd.DataFrame(nuke_verdict).set_index("scenario")
nuke_df.to_csv(TABLES_OUT / "nuclear_verdict_by_scenario.csv")
print(nuke_df.to_string())
print()
for scen, d in scenarios_data.items():
    if "Nuclear" not in list(map(str, d["inp"].coords["conversion_tech"].values)): continue
    sel = d["sol"]["operation_conversion_power_capacity"].sel(conversion_tech="Nuclear")
    by_c = {str(a): float(sel.sel(area=a).sum())/1000 for a in sel.coords["area"].values}
    nz = {k: v for k, v in by_c.items() if v > 0.01}
    if nz:
        print(f"  {scen}: " + ", ".join(f"{k}={v:.1f}GW" for k,v in sorted(nz.items(), key=lambda x: -x[1])))


## 6.5. Ramping diagnostic (ex-post)

The current solves run with `RAMPING_ENABLED = False` for tractability —
the LP can ramp any dispatchable tech 0↔full hour-to-hour. This section
asks: **did the LP actually need fast ramps?** We compute the empirical
hourly ramp rate from `operation_conversion_power` and compare against
the realistic limits in `RAMP_RATES_BASE`.

**Verdict thresholds (cap-weighted share of hours exceeding limit):**
- < 0.1% → **OK**: ramping constraints would not bind, result robust
- 0.1–5% → **MILD**: rare excursions, minor caveat in article
- 5–20% → **MODERATE**: sensitivity solve worth considering
- > 20% → **MAJOR**: result materially biased, redo with ramping enabled

Realistic limits (from `scripts/run_adequacy.py:RAMP_RATES_BASE`):
- Nuclear: 5%/h (conservative French fleet)
- Gas (CCGT): 50%/h (modern)
- Hydrogen_power_plant: 50%/h (H₂-CCGT)


In [ ]:
RAMP_REALISTIC = {
    "Nuclear":              0.05,
    "Gas":                  0.50,
    "Hydrogen_power_plant": 0.50,
}

ramping_rows = []
for scen, d in scenarios_data.items():
    sol = d["sol"]
    if "operation_conversion_power" not in sol.data_vars:
        continue
    op = sol["operation_conversion_power"].squeeze("year_op", drop=True)         # (area, tech, hour)
    cap = sol["operation_conversion_power_capacity"].squeeze("year_op", drop=True)  # (area, tech)
    techs_present = list(map(str, op.coords["conversion_tech"].values))

    for tech, realistic in RAMP_REALISTIC.items():
        if tech not in techs_present: continue
        disp = op.sel(conversion_tech=tech)      # (area, hour)
        capt = cap.sel(conversion_tech=tech)     # (area,)
        for area in disp.coords["area"].values:
            c_mw = float(capt.sel(area=area))
            if c_mw <= 1.0:  # < 1 MW deployed → skip
                continue
            hourly = disp.sel(area=area).values  # length 8760
            deltas = np.abs(np.diff(hourly))     # length 8759
            ramp_rates = deltas / c_mw
            ramping_rows.append({
                "scenario": scen,
                "tech": tech,
                "area": str(area),
                "capacity_GW": c_mw / 1000.0,
                "realistic_limit": realistic,
                "share_above_limit": float((ramp_rates > realistic).mean()),
                "p50_ramp": float(np.percentile(ramp_rates, 50)),
                "p95_ramp": float(np.percentile(ramp_rates, 95)),
                "p99_ramp": float(np.percentile(ramp_rates, 99)),
                "max_ramp": float(ramp_rates.max()),
            })

ramp_df = pd.DataFrame(ramping_rows)
if not ramp_df.empty:
    ramp_df.to_csv(TABLES_OUT / "ramping_by_area.csv", index=False)
    print(f"  Computed ramping for {len(ramp_df)} (scenario × tech × area) rows")
else:
    print("  No ramping data (operation_conversion_power missing or no qualifying techs)")


In [ ]:
# Capacity-weighted summary per (scenario, tech)
if not ramp_df.empty:
    def cw(group, col):
        w = group["capacity_GW"]
        return float((group[col] * w).sum() / w.sum()) if w.sum() > 0 else float("nan")

    summary_rows = []
    for (scen, tech), g in ramp_df.groupby(["scenario", "tech"]):
        summary_rows.append({
            "scenario": scen,
            "tech": tech,
            "total_capacity_GW": g["capacity_GW"].sum(),
            "realistic_limit": g["realistic_limit"].iloc[0],
            "cw_share_above_limit": cw(g, "share_above_limit"),
            "cw_p95_ramp":          cw(g, "p95_ramp"),
            "cw_p99_ramp":          cw(g, "p99_ramp"),
            "max_ramp_anywhere":    g["max_ramp"].max(),
        })
    ramp_summary = pd.DataFrame(summary_rows).set_index(["scenario", "tech"])
    ramp_summary.to_csv(TABLES_OUT / "ramping_summary.csv")
    print(ramp_summary.round(3).to_string())

    # Verdict
    print()
    print("=== RAMPING VERDICT (cap-weighted share of hours exceeding realistic ramp limit) ===")
    for (scen, tech), row in ramp_summary.iterrows():
        sh = row["cw_share_above_limit"]
        if sh < 0.001:    verdict = "OK"
        elif sh < 0.05:   verdict = "MILD"
        elif sh < 0.20:   verdict = "MODERATE"
        else:             verdict = "MAJOR"
        print(f"  {scen:30s} | {tech:22s} | share={sh:.2%} | p95={row['cw_p95_ramp']:.2%}/h | p99={row['cw_p99_ramp']:.2%}/h | max={row['max_ramp_anywhere']:.2f} | {verdict}")


In [ ]:
# Figure: bar chart of share-above-limit per (scenario, tech)
if not ramp_df.empty:
    pivot = ramp_summary.reset_index().pivot(
        index="scenario", columns="tech", values="cw_share_above_limit"
    )
    fig, ax = plt.subplots(figsize=(13, 5))
    pivot.plot.bar(ax=ax)
    ax.axhline(0.001, color="gray",    linestyle=":",  alpha=0.5, label="OK threshold (0.1%)")
    ax.axhline(0.05,  color="orange",  linestyle="--", label="MILD/MODERATE (5%)")
    ax.axhline(0.20,  color="red",     linestyle="--", label="MAJOR (20%)")
    ax.set_ylabel("Cap-weighted share of hours\nexceeding realistic ramp limit")
    ax.set_title("Ex-post ramping diagnostic — does the LP rely on unrealistic fast ramps?")
    ax.set_yscale("symlog", linthresh=0.001)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=45)
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    out = FIGS_OUT / "ramping_violation_share"
    fig.savefig(f"{out}.png", dpi=120); fig.savefig(f"{out}.svg"); plt.close()
    print(f"→ wrote {out}.{{png,svg}}")


## 7. Corridor expansion analysis

In [ ]:
CORRIDOR_LINKS = ["link_DK_DE", "link_NO_DK", "link_NL_DE", "link_FR_ES"]
corr_rows = []
for scen, d in scenarios_data.items():
    tpc = d["sol"]["operation_transport_power_capacity"]
    links_present = list(map(str, tpc.coords["link"].values))
    techs_present = list(map(str, tpc.coords["transport_tech"].values))
    if "electric_line" not in techs_present: continue
    sel = tpc.sel(transport_tech="electric_line")
    for link in CORRIDOR_LINKS:
        if link in links_present:
            mw = float(sel.sel(link=link).sum())
            corr_rows.append({"scenario": scen, "link": link, "built_GW": mw / 1000.0})
if corr_rows:
    corr_df = pd.DataFrame(corr_rows).pivot(index="link", columns="scenario", values="built_GW").fillna(0)
    base_col = next((c for c in ("R0_v1", "policy_re", "policy_nuke") if c in corr_df.columns), corr_df.columns[0])
    for col in corr_df.columns:
        if col != base_col and "_corr" in col:
            corr_df[f"Δ_{col}_vs_{base_col}"] = corr_df[col] - corr_df[base_col]
    corr_df.to_csv(TABLES_OUT / "corridor_built_capacity.csv")
    print(corr_df.round(2).to_string())


## 8. System cost summary

In [ ]:
cost_rows = []
for scen, d in scenarios_data.items():
    sol = d["sol"]
    row = {"scenario": scen}
    for var in ("annualised_totex",
                "operation_load_shedding_costs", "operation_spillage_costs",
                "operation_conversion_costs", "operation_storage_costs",
                "operation_transport_costs",
                "planning_conversion_costs", "planning_storage_costs",
                "planning_transport_costs"):
        if var in sol.data_vars:
            row[f"{var}_Geur"] = float(sol[var].sum()) / 1e9
    cost_rows.append(row)
cost_df = pd.DataFrame(cost_rows).set_index("scenario")
cost_df.to_csv(TABLES_OUT / "cost_breakdown.csv")
print(cost_df.round(2).to_string())


## 9. Sanity assertions

In [ ]:
def check(cond, msg):
    print(("  OK:  " if cond else "  FAIL: ") + msg)

if not headline_df.empty:
    for scen in headline_df.index:
        if "electrolyser_GW" in headline_df.columns:
            e = headline_df.loc[scen, "electrolyser_GW"]
            check(e > 0, f"{scen}: electrolyser_GW = {e:.1f} > 0")
        if "shed_hydrogen_TWh" in headline_df.columns:
            shed = headline_df.loc[scen, "shed_hydrogen_TWh"]
            check(shed < 0.01, f"{scen}: H₂ shed = {shed:.4f} TWh < 0.01")
    if "policy_re" in headline_df.index and "R0_v1" in headline_df.index:
        check(headline_df.loc["policy_re", "electrolyser_GW"] >= headline_df.loc["R0_v1", "electrolyser_GW"],
              "policy_re electrolyser ≥ R0_v1")

print()
print("=== DONE ===")
print(f"  CSVs in {TABLES_OUT}")
print(f"  Figures in {FIGS_OUT}")
print(f"  Loaded scenarios: {SCENARIOS}")
